In [43]:
import pandas as pd
df = pd.read_csv('customer_shopping_behavior.csv')
df.head()

,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,Yes,23,Credit Card,Weekly
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,Yes,49,PayPal,Weekly
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,Yes,31,PayPal,Annually


In [44]:
df.info

<bound method DataFrame.info of       Customer ID  Age  Gender Item Purchased     Category  \
0               1   55    Male         Blouse     Clothing   
1               2   19    Male        Sweater     Clothing   
2               3   50    Male          Jeans     Clothing   
3               4   21    Male        Sandals     Footwear   
4               5   45    Male         Blouse     Clothing   
...           ...  ...     ...            ...          ...   
3895         3896   40  Female         Hoodie     Clothing   
3896         3897   52  Female       Backpack  Accessories   
3897         3898   46  Female           Belt  Accessories   
3898         3899   44  Female          Shoes     Footwear   
3899         3900   52  Female        Handbag  Accessories   

      Purchase Amount (USD)       Location Size      Color  Season  \
0                        53       Kentucky    L       Gray  Winter   
1                        64          Maine    L     Maroon  Winter   
2            

In [3]:
df.describe()

,Customer ID,Age,Purchase Amount (USD),Review Rating,Previous Purchases
count,3900.000000,3900.000000,3900.000000,3863.000000,3900.000000
mean,1950.500000,44.068462,59.764359,3.750065,25.351538
std,1125.977353,15.207589,23.685392,0.716983,14.447125
min,1.000000,18.000000,20.000000,2.500000,1.000000
25%,975.750000,31.000000,39.000000,3.100000,13.000000
50%,1950.500000,44.000000,60.000000,3.800000,25.000000
75%,2925.250000,57.000000,81.000000,4.400000,38.000000
max,3900.000000,70.000000,100.000000,5.000000,50.000000


In [30]:
df.columns = df.columns.str.lower()
df.columns = df.columns.str.replace(' ','_')
df = df.rename(columns={'purchase_amount':'purchase_amount'})

In [31]:
df.columns

Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchase_amount_(usd)', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'promo_code_used', 'previous_purchases',
       'payment_method', 'frequency_of_purchases'],
      dtype='object')

In [32]:
#create a column age_group 
labels = ['young adult' , 'adult' , 'middle-aged', 'senior']
df['age_group'] = pd.qcut(df['age'], q=4, labels = labels)

In [33]:
df[['age' , 'age_group']].head(10)

,age,age_group
0,55,middle-aged
1,19,young adult
2,50,middle-aged
3,21,young adult
4,45,middle-aged
5,46,middle-aged
6,63,senior
7,27,young adult
8,26,young adult
9,57,middle-aged


In [34]:
# create column purchase_frequency_days

frequency_mapping = {
    'Fortnightly': 14,
    'Weekly': 7,
    'Monthly': 30,
    'Quarterly': 90,
    'Bi-Weekly': 14,
    'Annually': 365,
    'Every 3 Months': 90
}

df['purchase_frequency_days'] = df['frequency_of_purchases'].map(frequency_mapping)
        

In [35]:
df[['purchase_frequency_days','frequency_of_purchases']].head(10)

,purchase_frequency_days,frequency_of_purchases
0,14,Fortnightly
1,14,Fortnightly
2,7,Weekly
3,7,Weekly
4,365,Annually
5,7,Weekly
6,90,Quarterly
7,7,Weekly
8,365,Annually
9,90,Quarterly


In [36]:
df[['discount_applied','promo_code_used']].head(10)

,discount_applied,promo_code_used
0,Yes,Yes
1,Yes,Yes
2,Yes,Yes
3,Yes,Yes
4,Yes,Yes
5,Yes,Yes
6,Yes,Yes
7,Yes,Yes
8,Yes,Yes
9,Yes,Yes


In [37]:
(df['discount_applied'] == df['promo_code_used']).all()


np.True_

In [38]:
df = df.drop('promo_code_used', axis=1)

In [39]:
df.columns

Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchase_amount_(usd)', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'previous_purchases', 'payment_method',
       'frequency_of_purchases', 'age_group', 'purchase_frequency_days'],
      dtype='object')

In [40]:
from sqlalchemy import create_engine
import psycopg2

print("Packages imported successfully!")


Packages imported successfully!


In [41]:
pip install psycopg2-binary sqlalchemy

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [42]:
from sqlalchemy import create_engine, text

# Step 1: Connect to PostgreSQL (using default 'postgres' database first)
username = "postgres"      # default user
password = "akshitanayak"       # the password you set during installation
host = "localhost"         # if running locally
port = "5432"              # default PostgreSQL port
default_database = "postgres"   # connect to default database first

# Create engine for default database
default_engine = create_engine(f"postgresql+psycopg2://{username}:{password}@{host}:{port}/{default_database}")

# Step 2: Create the new database if it doesn't exist
database = "customer_shopping_behavior"   # the database you want to create

with default_engine.connect() as conn:
    # Check if database exists, if not create it
    conn.execute(text("COMMIT"))  # End any existing transaction
    try:
        conn.execute(text(f"CREATE DATABASE {database}"))
        print(f"Database '{database}' created successfully.")
    except Exception as e:
        if "already exists" in str(e):
            print(f"Database '{database}' already exists.")
        else:
            print(f"Error creating database: {e}")

# Step 3: Connect to the target database
engine = create_engine(f"postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}")

# Step 4: Load DataFrame into PostgreSQL
table_name = "customer"    # choose any table name
df.to_sql(table_name, engine, if_exists="replace", index=False)

print(f"Data successfully loaded into table '{table_name}' in database '{database}'.")

Database 'customer_shopping_behavior' already exists.
Data successfully loaded into table 'customer' in database 'customer_shopping_behavior'.


In [24]:

pip install pymysql sqlalchemy --user

Note: you may need to restart the kernel to use updated packages.


In [26]:
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy.exc import OperationalError
import pymysql

# MySQL connection parameters
username = "root"
password = "your_password"  # Replace with your actual password
host = "localhost"
port = "3306"
database = "customer_behavior"

try:
    # Create engine with connection verification
    engine = create_engine(f"mysql+pymysql://{username}:{password}@{host}:{port}/{database}")
    
    # Test the connection first
    with engine.connect() as connection:
        print("Connection successful!")
    
    # Write DataFrame to MySQL (assuming df is defined)
    table_name = "customer"
    df.to_sql(table_name, engine, if_exists="replace", index=False)
    print(f"Data written to table '{table_name}' successfully!")
    
    # Read back sample
    result = pd.read_sql("SELECT * FROM customer LIMIT 5;", engine)
    print(result)
    
except OperationalError as e:
    print(f"Connection failed: {e}")
    print("\nTroubleshooting steps:")
    print("1. Make sure MySQL server is running")
    print("2. Verify username and password are correct")
    print("3. Check if the database 'customer_behavior' exists")
    print("4. Confirm MySQL is running on port 3306")
    
except Exception as e:
    print(f"An error occurred: {e}")

Connection failed: (pymysql.err.OperationalError) (2003, "Can't connect to MySQL server on 'localhost' ([WinError 10061] No connection could be made because the target machine actively refused it)")
(Background on this error at: https://sqlalche.me/e/20/e3q8)

Troubleshooting steps:
1. Make sure MySQL server is running
2. Verify username and password are correct
3. Check if the database 'customer_behavior' exists
4. Confirm MySQL is running on port 3306


In [28]:
pip install pyodbc sqlalchemy 

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
